# B5T2 · Cuaderno de mando de la evaluación

Este cuaderno **conduce**; el código **vive en el paquete `agente/`**. La separación no es cosmética:

- El día 24 un evaluador clona el repo y ejecuta `evaluar("holdout.jsonl")` sin abrir ningún notebook. Si la lógica estuviera aquí, no funcionaría. Ese fue exactamente el problema del repo del 17-sep.
- Todo lo que se ve aquí se puede volver a generar con `python -m agente.cli …`. Este cuaderno es la misma máquina con ventanas.

Qué hace cada bloque:

| Bloque | Gasta API | Para qué |
|---|---|---|
| 1 · Preparación y **calentamiento** | no | comprobar corpus y clave, precargar el retrieval |
| 2 · Una pregunta | sí (1) | ver una trayectoria completa antes de lanzar 60 |
| 3 · Ejecutar una arquitectura | sí (20 × reps) | guarda un JSON por pregunta; **idempotente** |
| 4 · Leer la tabla | no | qué falla, y por qué, mirando la trayectoria guardada |
| 5 · Comparativa | no | la tabla del informe |
| 6 · Recall del retrieval | casi no | la tabla del §4.4 |

Cambia `ARQ` y vuelve a ejecutar los bloques 3-5 para cada peldaño:

`baseline` → `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas`

Cada peldaño **añade** al anterior y no quita nada, así la diferencia entre dos filas consecutivas se atribuye a una sola cosa.

## 1 · Preparación

**Por qué hay un calentamiento.** La primera vez que el agente llama a `search_filings`, Python carga el índice FAISS y el modelo de embeddings desde disco: entre 15 y 20 segundos. Si eso ocurre *dentro* de una pregunta, esos segundos se suman a su latencia y la columna del informe queda contaminada.

Y hay algo peor: el proceso del kernel sobrevive entre repeticiones, así que **solo la repetición 1 paga la carga**. En la tabla eso aparece como varianza entre repeticiones — precisamente lo que las tres repeticiones intentan medir. Un artefacto de arranque disfrazado de ruido del modelo.

`calentar()` lo saca fuera del cronómetro. `ejecutar()` la llama sola, pero conviene verla aquí para saber qué está pasando.

De paso, si el modelo ya está en la caché local, pone `HF_HUB_OFFLINE=1`: desaparece el aviso de peticiones anónimas a Hugging Face, se ahorra un viaje de red por arranque y —lo que de verdad importa— **el día 24 la ejecución no depende de que Hugging Face esté disponible**. En un clon recién hecho, sin caché, no se activa, para que la primera descarga funcione con normalidad.

In [1]:
# 1 · Preparación ---------------------------------------------------------------
# autoreload: si editas un fichero de agente/, el cuaderno lo recoge sin reiniciar.
%load_ext autoreload
%autoreload 2

import os, json, warnings
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# tqdm quiere dibujar una barra de progreso interactiva y necesita `ipywidgets`.
# Sin él se cae a la barra de texto, que funciona igual. Silenciamos el aviso.
# (Si prefieres las barras bonitas: `uv add --dev ipywidgets` y quita esta línea.)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

from agente.config import ARQUITECTURAS, MODELO, REPETICIONES
from agente.corpus import dir_corpus, cargar_xbrl, raiz_repo
from agente.interfaz import (responder, ejecutar, puntuar, comparar, resumir, calentar,
                             dir_rep, dir_arquitectura, dir_resultados, leer_golden)
from agente.resultado import pretty_trace

# --- lo que vamos a evaluar en esta pasada ---
ARQ    = "a4_comparativas"                 # baseline · a1_guardrails · a2_retrieval · a3_hibrido · a4_comparativas
REPS   = REPETICIONES               # 3
GOLDEN = "data/golden_set.jsonl"

# --- comprobaciones, sin gastar nada ---
print("raíz del repo :", raiz_repo())
print("corpus        :", dir_corpus(), "·", len(cargar_xbrl()), "hechos XBRL")
print("modelo        :", MODELO)
print("clave OpenRouter en el entorno:", bool(os.environ.get("OPENROUTER_API_KEY")))   # solo sí/no, nunca el valor
HAY_CLAVE = bool(os.environ.get("OPENROUTER_API_KEY"))

golden = leer_golden(GOLDEN)
print(f"golden set    : {len(golden)} preguntas · "
      f"{sum(1 for g in golden if g.get('ancla_texto'))} con ancla")

# --- calentamiento: fuera del cronómetro, una sola vez por kernel ---
print("\ncalentando el retrieval…")
calentar()
print("HF_HUB_OFFLINE =", os.environ.get("HF_HUB_OFFLINE", "(sin poner)"))

# Las arquitecturas, como tabla: qué enciende cada una
display(pd.DataFrame([a.como_dict() for n, a in ARQUITECTURAS.items() if n != "final"])
        .set_index("nombre")[["limites","verificador_cifras","verificador_cita","esquema_estricto",
                              "prompt","filtros_forzados","reescritura","hibrido","descripcion"]])

raíz del repo : C:\dev\MIAX_Tareas\B5T2-agentes
corpus        : C:\dev\MIAX_Tareas\B5T2-agentes\corpus · 135 hechos XBRL
modelo        : openrouter:google/gemini-3.8-flash
clave OpenRouter en el entorno: True
golden set    : 20 preguntas · 14 con ancla

calentando el retrieval…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3547.09it/s]

  retrieval precargado: 1749 vectores, 1749 fragmentos (5.4 s)
HF_HUB_OFFLINE = 1


,limites,verificador_cifras,verificador_cita,esquema_estricto,prompt,filtros_forzados,reescritura,hibrido,descripcion
nombre,,,,,,,,,
baseline,False,False,False,False,base,False,False,False,"El agente del día 10, tal cual. Congelado antes de tocar..."
a1_guardrails,True,True,True,True,honesto,False,False,False,"+ límites de llamadas, verificador XBRL, verificador de ..."
a2_retrieval,True,True,True,True,honesto,True,True,False,+ filtros por metadatos forzados y reescritura de la con...
a3_hibrido,True,True,True,True,honesto,True,True,True,"+ híbrido BM25 con RRF, medido DESPUÉS de la reescritura."
a4_comparativas,True,True,True,True,comparativas,True,True,True,+ procedimiento explícito de comparativas y conceptos po...


### Limpiar antes de empezar

`ejecutar()` es **idempotente**: si ya existe `crudo/<id>.json`, salta la pregunta. Eso permite relanzar una ejecución cortada sin repetir lo hecho — pero también significa que **una medición mala se queda guardada** hasta que la borres.

- `BORRAR_TODO = True` deja la arquitectura a cero (crudo, tablas y resumen).
- `BORRAR_REP = n` borra solo esa repetición.
- Con los dos apagados, la celda **solo informa** de lo que hay.

Acuérdate de volver a dejarlos apagados después de borrar, o la próxima vez que pases por aquí te llevas por delante lo bueno.

In [6]:
BORRAR_REP  = None           # p. ej. 1 para rehacer solo la repetición 1 de ARQ
BORRAR_TODO = False          # True para dejar ARQ completamente a cero

import shutil

if BORRAR_TODO:
    carpeta = dir_arquitectura(ARQ)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada entera: {carpeta}")
elif BORRAR_REP is not None:
    carpeta = dir_rep(ARQ, BORRAR_REP)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada: {carpeta}")
else:
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        n = len(list((carpeta / "crudo").glob("*.json"))) if (carpeta / "crudo").is_dir() else 0
        print(f"{carpeta.name}: {n} preguntas guardadas")
    print("(nada borrado)")

(nada borrado)


## 2 · Una pregunta suelta

Antes de lanzar 60 llamadas, una. Se ve la trayectoria entera: qué herramienta pidió, con qué argumentos, qué le devolvió, y la respuesta estructurada. Es la misma `pretty_trace` de clase.

In [2]:
PREGUNTA = "¿Cuál fue el revenue de NVIDIA en FY2025?"

if HAY_CLAVE:
    r = responder(PREGUNTA, thread_id="suelta", arquitectura=ARQ)
    print(pretty_trace(r))
    print(f"\n  [{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
else:
    print("Sin clave: este bloque no se ejecuta.")

  1. get_xbrl_fact(ticker='NVDA', fiscal_year=2025, concept='Revenues')
       -> NVDA FY2025 · Revenues = 130,497,000,000 USD (cierre de ejercicio 2025-01-26, según el 10-K)
       -> Returning structured response: respuesta='El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2025 (FY2025) fue de 130.497.000.000 USD.' cifra=130497000000.0 unidad='USD' ticker='NVDA' ejercicio=2025 fuente='x…

  respuesta: El revenue (ingresos totales) de NVIDIA en el ejercicio fiscal 2025 (FY2025) fue de 130.497.000.000 USD.
  cifra: 130497000000.0 USD · fuente: xbrl · chunk: None

  [14.0 s · 0.42 ¢]


## 3 · Ejecutar una arquitectura, `REPS` veces

`ejecutar()` guarda **un JSON por pregunta** en `resultados/agente/<ARQ>/rep<n>/crudo/` con la trayectoria completa. Si esto se corta a mitad, vuelve a ejecutar la celda y sigue donde estaba.

`puntuar()` no gasta API: lee esos JSON, aplica los tres evaluadores y escribe `tabla.csv`. Como está separado, si mañana corregimos un evaluador se re-puntúa todo en segundos.

El `thread_id` lleva la repetición dentro (`baseline-rep2-gX-013`): sin eso, la repetición 2 vería la conversación de la 1 en el checkpointer.

In [3]:
tablas = {}
if HAY_CLAVE:
    for rep in range(1, REPS + 1):
        print(f"\n== {ARQ} · repetición {rep}/{REPS} ==")
        ejecutar(GOLDEN, ARQ, rep)                 # API · idempotente · calienta antes del bucle
        tablas[rep] = puntuar(ARQ, rep)            # sin API · incluye recall@5
        display(resumir(tablas[rep], f"{ARQ} rep{rep}"))
else:
    # Sin clave se puede puntuar lo que ya esté guardado de otras veces.
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        if (carpeta / "crudo").is_dir() and any((carpeta / "crudo").glob("*.json")):
            tablas[int(carpeta.name[3:])] = puntuar(ARQ, int(carpeta.name[3:]), con_recall=False)
    print("Sin clave: puntuado lo guardado →", list(tablas) or "nada")


== a4_comparativas · repetición 1/3 ==
  retrieval precargado: 1749 vectores, 1749 fragmentos + BM25 (0.2 s)
  [1/20] gX-001 … 11.5s · 0.45¢ · 1 llamadas · fuente=xbrl
  [2/20] gX-002 … 16.0s · 0.69¢ · 2 llamadas · fuente=xbrl
  [3/20] gX-003 … 15.5s · 0.63¢ · 2 llamadas · fuente=xbrl
  [4/20] gX-004 … 9.4s · 0.64¢ · 2 llamadas · fuente=xbrl
  [5/20] gX-005 … 8.5s · 0.71¢ · 2 llamadas · fuente=xbrl
  [6/20] gX-006 … 8.1s · 0.43¢ · 1 llamadas · fuente=xbrl
  [7/20] gX-007 … 129.1s · 4.31¢ · 5 llamadas · fuente=texto
  [8/20] gX-008 … 25.7s · 1.00¢ · 2 llamadas · fuente=texto
  [9/20] gX-009 … error del proveedor (400), reintento … 50.1s · 2.17¢ · 5 llamadas · fuente=None
  [10/20] gX-010 … 31.9s · 0.98¢ · 2 llamadas · fuente=texto
  [11/20] gX-011 … 13.1s · 0.93¢ · 2 llamadas · fuente=texto
  [12/20] gX-012 … 20.7s · 0.63¢ · 1 llamadas · fuente=texto
  [13/20] gX-013 … 68.6s · 1.94¢ · 7 llamadas · fuente=ambas
  [14/20] gX-014 … 20.4s · 1.19¢ · 3 llamadas · fuente=ambas
  [15/20] gX-01

{'versión': 'a4_comparativas rep1',
 'acierto': 0.95,
 'cita': 0.9285714285714286,
 'cifra': 1.0,
 'trayectoria': 1.0,
 'recall@5': 0.8571428571428571,
 'coste medio (¢)': 1.4565937500000001,
 'latencia media (s)': 40.273400159971786,
 'llamadas/pregunta': 3.4,
 '% fuente=ninguna': 0.0,
 'errores': 0,
 'reintentos proveedor': 1,
 '% corrigió cifra': 0.0,
 '% corrigió cita': 0.0,
 'reintentos esquema': 0.1,
 '% límite alcanzado': 0.0,
 '% búsquedas reescritas': np.float64(0.9130434782608695),
 '% búsquedas con ticker': 1.0,
 'acierto numerica': 1.0,
 'acierto extractiva': 0.8333333333333334,
 'acierto comparativa': 1.0}


== a4_comparativas · repetición 2/3 ==
  retrieval precargado: 1749 vectores, 1749 fragmentos + BM25 (0.0 s)
  [1/20] gX-001 … 13.2s · 0.45¢ · 1 llamadas · fuente=xbrl
  [2/20] gX-002 … 18.0s · 0.67¢ · 2 llamadas · fuente=xbrl
  [3/20] gX-003 … 23.7s · 0.62¢ · 2 llamadas · fuente=xbrl
  [4/20] gX-004 … 33.4s · 0.71¢ · 2 llamadas · fuente=xbrl
  [5/20] gX-005 … 9.1s · 0.63¢ · 2 llamadas · fuente=xbrl
  [6/20] gX-006 … 6.4s · 0.43¢ · 1 llamadas · fuente=xbrl
  [7/20] gX-007 … 111.6s · 2.89¢ · 5 llamadas · fuente=texto
  [8/20] gX-008 … 26.4s · 1.15¢ · 2 llamadas · fuente=texto
  [9/20] gX-009 … 104.1s · 2.72¢ · 4 llamadas · fuente=texto
  [10/20] gX-010 … 42.2s · 0.98¢ · 2 llamadas · fuente=texto
  [11/20] gX-011 … 117.9s · 2.04¢ · 5 llamadas · fuente=texto
  [12/20] gX-012 … 12.7s · 0.99¢ · 2 llamadas · fuente=texto
  [13/20] gX-013 … 112.4s · 3.46¢ · 9 llamadas · fuente=ambas
  [14/20] gX-014 … 59.0s · 1.14¢ · 3 llamadas · fuente=ambas
  [15/20] gX-015 … 80.5s · 2.95¢ · 7 llamadas · f

{'versión': 'a4_comparativas rep2',
 'acierto': 1.0,
 'cita': 1.0,
 'cifra': 1.0,
 'trayectoria': 1.0,
 'recall@5': 0.8571428571428571,
 'coste medio (¢)': 1.5543975,
 'latencia media (s)': 51.930601519951594,
 'llamadas/pregunta': 3.7,
 '% fuente=ninguna': 0.0,
 'errores': 0,
 'reintentos proveedor': 0,
 '% corrigió cifra': 0.0,
 '% corrigió cita': 0.0,
 'reintentos esquema': 0.0,
 '% límite alcanzado': 0.05,
 '% búsquedas reescritas': np.float64(0.7241379310344828),
 '% búsquedas con ticker': 1.0,
 'acierto numerica': 1.0,
 'acierto extractiva': 1.0,
 'acierto comparativa': 1.0}


== a4_comparativas · repetición 3/3 ==
  retrieval precargado: 1749 vectores, 1749 fragmentos + BM25 (0.0 s)
  [1/20] gX-001 … 47.3s · 0.42¢ · 1 llamadas · fuente=xbrl
  [2/20] gX-002 … 10.6s · 0.65¢ · 2 llamadas · fuente=xbrl
  [3/20] gX-003 … 16.4s · 0.70¢ · 2 llamadas · fuente=xbrl
  [4/20] gX-004 … 22.0s · 0.64¢ · 2 llamadas · fuente=xbrl
  [5/20] gX-005 … 11.9s · 0.78¢ · 2 llamadas · fuente=xbrl
  [6/20] gX-006 … 7.0s · 0.47¢ · 1 llamadas · fuente=xbrl
  [7/20] gX-007 … 143.3s · 3.45¢ · 7 llamadas · fuente=texto
  [8/20] gX-008 … 29.9s · 1.53¢ · 3 llamadas · fuente=texto
  [9/20] gX-009 … 105.3s · 2.70¢ · 4 llamadas · fuente=texto
  [10/20] gX-010 … 47.3s · 1.00¢ · 2 llamadas · fuente=texto
  [11/20] gX-011 … 51.6s · 1.97¢ · 4 llamadas · fuente=texto
  [12/20] gX-012 … 77.0s · 1.27¢ · 3 llamadas · fuente=texto
  [13/20] gX-013 … 84.3s · 1.73¢ · 6 llamadas · fuente=ambas
  [14/20] gX-014 … 39.4s · 1.90¢ · 3 llamadas · fuente=ambas
  [15/20] gX-015 … 179.1s · 3.51¢ · 9 llamadas · f

{'versión': 'a4_comparativas rep3',
 'acierto': 0.95,
 'cita': 0.9285714285714286,
 'cifra': 0.9285714285714286,
 'trayectoria': 1.0,
 'recall@5': 0.8571428571428571,
 'coste medio (¢)': 1.5516787499999998,
 'latencia media (s)': 55.28153782999143,
 'llamadas/pregunta': 3.7,
 '% fuente=ninguna': 0.0,
 'errores': 0,
 'reintentos proveedor': 1,
 '% corrigió cifra': 0.0,
 '% corrigió cita': 0.1,
 'reintentos esquema': 0.0,
 '% límite alcanzado': 0.05,
 '% búsquedas reescritas': np.float64(0.8275862068965517),
 '% búsquedas con ticker': 1.0,
 'acierto numerica': 1.0,
 'acierto extractiva': 1.0,
 'acierto comparativa': 0.875}

## 4 · Leer la tabla: qué falla y por qué

Una fila por pregunta. Las columnas que importan:

- **`acierto`**: todos los evaluadores aplicables en `True`.
- **`cita` / `cifra` / `trayectoria`**: los tres del enunciado. Vacío = no aplica a esa familia.
- **`herramientas`**: el camino que siguió. Una numérica sin `get_xbrl_fact` suspende trayectoria aunque la cifra sea correcta.
- **`fuente`**: si dice `ninguna` en una pregunta con respuesta, no la encontró; si dice `xbrl` con una cifra que no cuadra, la leyó de donde no debía o la inventó.
- **`recall5` / `pos_ancla`**: el retriever, con los filtros del golden. `pos_ancla=6` y `pos_ancla=900` fallan igual el recall@5 y no son el mismo problema.

In [4]:
REP_A_MIRAR = 1
if REP_A_MIRAR in tablas:
    t = tablas[REP_A_MIRAR]

    # `cifra`/`cita`/`trayectoria` son los VEREDICTOS; `cifra_dada`/`cita_dada`,
    # lo que respondió el agente. Ver los dos juntos es lo que delata si un fallo
    # es de capacidad o de convención (p. ej. poner la variación donde se espera
    # el valor del ejercicio).
    cols = ["id","familia","acierto","cita","cifra","trayectoria","fuente",
            "cifra_dada","cifra_esperada","ratio_cifra","herramientas",
            "n_llamadas","coste_usd","latencia_s","recall5","pos_ancla"]
    cols = [c for c in cols if c in t]

    def colorear(fila):
        return ["background-color:#fde2e2" if fila.get("acierto") is False else
                ("background-color:#e2f5e2" if fila.get("acierto") is True else "") for _ in fila]

    display(t[cols].style.apply(colorear, axis=1)
            .format({"coste_usd": "{:.4f}", "latencia_s": "{:.1f}",
                     "cifra_dada": "{:,.0f}", "cifra_esperada": "{:,.0f}",
                     "ratio_cifra": "{:.3f}"}, na_rep="—"))

    fallos = t[t["acierto"] == False]                     # noqa: E712
    print(f"\n{len(fallos)} fallos de {len(t)}. Por familia:")
    display(t.groupby("familia")["acierto"].agg(["mean", "count"]).rename(columns={"mean": "tasa"}))

    # Instrumentación: qué guardrail actuó. Un guardrail que nunca salta no ha
    # aportado nada, y eso se cuenta en vez de estimarse con una ablación.
    instr = [c for c in ["corrigio_cifra","corrigio_cita","reintentos_esquema",
                         "limite_alcanzado","n_busquedas","busquedas_con_ticker",
                         "busquedas_con_item","uso_read_section"] if c in t]
    if instr:
        print("\nGuardrails y retrieval (media sobre las 20 preguntas):")
        display(t[instr].mean().to_frame("valor").T)

    print("\nLas 3 más lentas y las 3 más caras:")
    display(t.nlargest(3, "latencia_s")[["id","latencia_s","n_llamadas","herramientas"]])
    display(t.nlargest(3, "coste_usd")[["id","coste_usd","n_llamadas","herramientas"]])
else:
    print("No hay tabla para esa repetición.")

,id,familia,acierto,cita,cifra,trayectoria,fuente,cifra_dada,cifra_esperada,ratio_cifra,herramientas,n_llamadas,coste_usd,latencia_s,recall5,pos_ancla
0,gX-001,numerica,True,—,True,True,xbrl,"130,497,000,000","130,497,000,000",1.000,get_xbrl_fact,1,0.0045,11.5,—,—
1,gX-002,numerica,True,—,True,True,xbrl,"112,010,000,000","112,010,000,000",1.000,list_available → get_xbrl_fact,2,0.0069,16.0,—,—
2,gX-003,numerica,True,—,True,True,xbrl,"402,836,000,000","402,836,000,000",1.000,list_available → get_xbrl_fact,2,0.0063,15.5,—,—
3,gX-004,numerica,True,—,True,True,xbrl,"57,372,000,000","57,372,000,000",1.000,list_available → get_xbrl_fact,2,0.0064,9.4,—,—
4,gX-005,numerica,True,—,True,True,xbrl,"139,514,000,000","139,514,000,000",1.000,list_available → get_xbrl_fact,2,0.0071,8.5,—,—
5,gX-006,numerica,True,—,True,True,xbrl,"243,686,000,000","243,686,000,000",1.000,get_xbrl_fact,1,0.0043,8.1,—,—
6,gX-007,extractiva,True,True,—,True,texto,—,—,—,list_available → search_filings → search_filings → search_filings → search_filings,5,0.0431,129.1,False,15.000000
7,gX-008,extractiva,True,True,—,True,texto,—,—,—,list_available → search_filings,2,0.0100,25.7,True,3.000000
8,gX-009,extractiva,False,False,—,True,—,—,—,—,list_available → search_filings → search_filings → search_filings → search_filings,5,0.0217,50.1,True,1.000000
9,gX-010,extractiva,True,True,—,True,texto,—,—,—,list_available → search_filings,2,0.0098,31.9,True,5.000000



1 fallos de 20. Por familia:


,tasa,count
familia,,
comparativa,1.000000,8
extractiva,0.833333,6
numerica,1.000000,6



Guardrails y retrieval (media sobre las 20 preguntas):


,corrigio_cifra,corrigio_cita,reintentos_esquema,limite_alcanzado,n_busquedas,busquedas_con_ticker,busquedas_con_item,uso_read_section
valor,0.0,0.0,0.1,0.0,1.15,1.15,1.15,0.0



Las 3 más lentas y las 3 más caras:


,id,latencia_s,n_llamadas,herramientas
6,gX-007,129.058004,5,list_available → search_filings → search_filings → searc...
18,gX-019,110.706820,5,list_available → get_xbrl_fact → get_xbrl_fact → search_...
15,gX-016,86.604345,7,list_available → get_xbrl_fact → get_xbrl_fact → get_xbr...


,id,coste_usd,n_llamadas,herramientas
6,gX-007,0.043132,5,list_available → search_filings → search_filings → searc...
18,gX-019,0.031890,5,list_available → get_xbrl_fact → get_xbrl_fact → search_...
19,gX-020,0.026310,7,list_available → get_xbrl_fact → get_xbrl_fact → get_xbr...


### Las trayectorias de los fallos

Esto es lo que en clase había que imprimir a mano. Aquí se lee de lo guardado: **no gasta nada**. Para cada fallo, la pregunta, lo esperado, la trayectoria y la respuesta.

In [5]:
def ver_traza(arq, rep, id_):
    reg = json.loads((dir_rep(arq, rep) / "crudo" / f"{id_}.json").read_text(encoding="utf-8"))
    it = reg["item"]
    print("=" * 88)
    print(f"[{it['id']} · {it['familia']}] {it['pregunta']}")
    print(f"  esperado: {str(it.get('respuesta_esperada'))[:110]}")
    if it.get("herramienta_esperada"):
        print(f"  herramientas esperadas: {it['herramienta_esperada']}")
    print()
    print(reg.get("error") or pretty_trace(reg["resultado"]))

if REP_A_MIRAR in tablas:
    for id_ in tablas[REP_A_MIRAR].loc[tablas[REP_A_MIRAR]["acierto"] == False, "id"]:    # noqa: E712
        ver_traza(ARQ, REP_A_MIRAR, id_)

# Y cualquier otra, aunque haya acertado:
#   ver_traza(ARQ, 1, "gX-003")

[gX-009 · extractiva] ¿Qué obligaciones concretas dice Meta que le impone la DMA europea en FY2025?
  esperado: Restricciones sobre la combinación de datos entre servicios, sobre fusiones y adquisiciones y sobre el diseño 
  herramientas esperadas: ['search_filings']

  1. list_available()
       -> Contenido disponible en el corpus: - AAPL (Apple Inc.): ejercicios FY2024, FY2025; secciones disponibles: 1A, 7, 7A, 8. - AMZN (AMAZON COM INC): ejercicios FY2024, FY2025; secciones disponibles: 1A, 7, 7A, 8. - GOOGL (Al…
  2. search_filings(item='1A', fiscal_year=2025, ticker='META', query='Digital Markets Act DMA obligations gatekeeper', k=5)
       -> (consulta reescrita: 'Digital Markets Act" gatekeeper obligations')  [META-2025-1A-0001] META FY2025 Item 1A (similitud 0.032) Risks Related to Government Regulation and Enforcement  •government restrictions on access to…
  3. search_filings(k=3, ticker='META', item='7', fiscal_year=2025, query='Digital Markets Act DMA')
       -> (consulta

### Qué falla siempre y qué es ruido

El bloque anterior mira **una** repetición. Esta celda cruza las tres, y sirve para separar dos cosas que se confunden con facilidad:

- **falla SIEMPRE** (0 de 3) — un defecto reproducible. Esto sí se arregla con código: un guardrail, un filtro, una instrucción en el prompt. Es la lista de trabajo.
- **oscila** (1 o 2 de 3) — el modelo no es determinista y a veces toma otro camino. No se arregla escribiendo código; es el **ruido de fondo** del sistema.
- **acierta siempre** (3 de 3) — resuelto. Lo que hay que vigilar aquí es no romperlo al subir de peldaño.

La segunda categoría es la que da sentido a haber repetido tres veces. Si A1 sube dos preguntas y en el baseline ya había dos que iban y venían solas, no has demostrado nada: la mejora cabe dentro del ruido. Cuántas oscilan es, en la práctica, la barra que cualquier peldaño posterior tiene que superar para ser creíble.

Debajo sale la **tasa por evaluador** sobre las 60 invocaciones, contando solo las preguntas a las que cada uno aplica —`cifra` no se evalúa en una extractiva, `cita` no se evalúa en una numérica—. Eso dice por dónde se pierde el acierto: si `cifra` va bien y `cita` mal, el problema es de trazabilidad, no de datos; si `trayectoria` va mal, el agente está acertando por el camino equivocado.

In [6]:
import pandas as pd
reps = sorted(tablas)
todo = pd.concat([t.assign(_rep=r) for r, t in tablas.items()])

m = todo.pivot_table(index=["familia","id"], columns="_rep", values="acierto", aggfunc="first")
m["aciertos"] = m[reps].fillna(False).sum(axis=1).astype(int)
m["veredicto"] = m["aciertos"].map(
    lambda n: "falla SIEMPRE" if n == 0 else ("acierta siempre" if n == len(reps) else "oscila"))
display(m.sort_values(["aciertos", "familia"]))
print(m["veredicto"].value_counts().to_string())

print("\nTasa por evaluador sobre las 3 reps (solo donde aplica):")
for ev in ["cita", "cifra", "trayectoria"]:
    s = todo[ev].dropna()
    print(f"  {ev:12s} {s.mean():.0%}  ({int(s.sum())}/{len(s)})")

_rep                    1     2      3  aciertos        veredicto
familia     id                                                   
comparativa gX-018   True  True  False         2           oscila
extractiva  gX-009  False  True   True         2           oscila
comparativa gX-013   True  True   True         3  acierta siempre
            gX-014   True  True   True         3  acierta siempre
            gX-015   True  True   True         3  acierta siempre
            gX-016   True  True   True         3  acierta siempre
            gX-017   True  True   True         3  acierta siempre
            gX-019   True  True   True         3  acierta siempre
            gX-020   True  True   True         3  acierta siempre
extractiva  gX-007   True  True   True         3  acierta siempre
            gX-008   True  True   True         3  acierta siempre
            gX-010   True  True   True         3  acierta siempre
            gX-011   True  True   True         3  acierta siempre
            gX-012   True  True   True         3  acierta siempre
numerica    gX-001   True  True   True         3  acierta siempre
            gX-002   True  True   True         3  acierta siempre
            gX-003   True  True   True         3  acierta siempre
            gX-004   True  True   True         3  acierta siempre
            gX-005   True  True   True         3  acierta siempre
            gX-006   True  True   True         3  acierta siempre

veredicto
acierta siempre    18
oscila              2

Tasa por evaluador sobre las 3 reps (solo donde aplica):
  cita         95%  (40/42)
  cifra        98%  (41/42)
  trayectoria  100%  (60/60)


## 5 · La comparativa: una fila por arquitectura

Media de las repeticiones, con mínimo y máximo para ver si un salto es mejora o ruido. `comparativa.md` es la tabla del informe con el mejor valor de cada columna en negrita.

In [7]:
comp = comparar()
cols = ["arquitectura","reps","acierto","acierto numerica","acierto extractiva","acierto comparativa",
        "cita","cifra","trayectoria","recall@5","coste medio (¢)","latencia media (s)","llamadas/pregunta"]
display(comp[[c for c in cols if c in comp]].round(3))

if len(comp) and "acierto min" in comp:
    print("\nRango entre repeticiones (acierto):")
    display(comp[["arquitectura","acierto min","acierto","acierto max"]].round(3))

md_path = dir_resultados() / "comparativa.md"
if md_path.is_file():
    display(Markdown(md_path.read_text(encoding="utf-8")))

,arquitectura,reps,acierto,acierto numerica,acierto extractiva,acierto comparativa,cita,cifra,trayectoria,recall@5,coste medio (¢),latencia media (s),llamadas/pregunta
0,baseline,3,0.617,1.0,0.944,0.083,0.524,0.500,0.833,0.500,1.615,20.329,3.950
1,a1_guardrails,3,0.750,1.0,1.000,0.375,0.643,1.000,0.750,0.500,1.409,17.147,3.250
2,a2_retrieval,3,0.750,1.0,1.000,0.375,0.643,1.000,0.767,0.643,1.443,29.401,3.483
3,a4_comparativas,3,0.967,1.0,0.944,0.958,0.952,0.976,1.000,0.857,1.521,49.162,3.600



Rango entre repeticiones (acierto):


,arquitectura,acierto min,acierto,acierto max
0,baseline,0.60,0.617,0.65
1,a1_guardrails,0.75,0.750,0.75
2,a2_retrieval,0.75,0.750,0.75
3,a4_comparativas,0.95,0.967,1.00


| arquitectura | reps | acierto | acierto numerica | acierto extractiva | acierto comparativa | cita | cifra | trayectoria | recall@5 | coste medio (¢) | latencia media (s) | llamadas/pregunta | % fuente=ninguna |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 3 | 61.7% | **100.0%** | 94.4% | 8.3% | 52.4% | 50.0% | 83.3% | 50.0% | 1.62 | 20.33 | 3.95 | 0.0% |
| a1_guardrails | 3 | 75.0% | **100.0%** | **100.0%** | 37.5% | 64.3% | **100.0%** | 75.0% | 50.0% | **1.41** | **17.15** | **3.25** | 0.0% |
| a2_retrieval | 3 | 75.0% | **100.0%** | **100.0%** | 37.5% | 64.3% | **100.0%** | 76.7% | 64.3% | 1.44 | 29.40 | 3.48 | 0.0% |
| a4_comparativas | 3 | **96.7%** | **100.0%** | 94.4% | **95.8%** | **95.2%** | 97.6% | **100.0%** | **85.7%** | 1.52 | 49.16 | 3.60 | 0.0% |

Media de las repeticiones. Mejor valor de cada columna en negrita. recall@5 sobre las preguntas con ancla, con los filtros del golden set. Coste y latencia por pregunta.

## 6 · El retrieval solo: recall@5 por configuración

La tabla del §4.4. No interviene el agente: se mide el buscador con los filtros del golden set. Cero llamadas de API salvo la reescritura de la consulta (una por pregunta, cacheada en `resultados/retrieval/reescrituras.json`).

`posiciones` dice en qué puesto quedó el ancla en cada configuración: es lo que explica **por qué** una configuración gana.

In [8]:
from agente.recall import medir_recall, CONFIGS

configs = dict(CONFIGS)
if not HAY_CLAVE:                       # sin clave no se puede reescribir
    configs = {k: v for k, v in configs.items() if "reescritura" not in k}

recall = medir_recall(configs)
display(recall)

pos = pd.read_csv(dir_resultados() / "retrieval" / "posiciones.csv")
display(pos.pivot(index="id", columns="config", values="pos_ancla"))

  1 · denso plano                  2/14  (14.3%)
  2 · + filtro de metadatos        7/14  (50.0%)
  3 · + híbrido BM25               7/14  (50.0%)
  4 · + reescritura de consulta    9/14  (64.3%)
  5 · reescritura + híbrido        12/14  (85.7%)


,configuración,recall@5,aciertos,coste
0,1 · denso plano,0.142857,2/14,0 llamadas al LLM
1,2 · + filtro de metadatos,0.500000,7/14,0 llamadas al LLM
2,3 · + híbrido BM25,0.500000,7/14,"0 llamadas, +1 índice en memoria"
3,4 · + reescritura de consulta,0.642857,9/14,1 llamada al LLM por búsqueda
4,5 · reescritura + híbrido,0.857143,12/14,1 llamada + índice léxico


config,1 · denso plano,2 · + filtro de metadatos,3 · + híbrido BM25,4 · + reescritura de consulta,5 · reescritura + híbrido
id,,,,,
gX-007,259,16,22,23,15
gX-008,49,2,2,10,3
gX-009,1,1,1,1,1
gX-010,1020,21,14,14,5
gX-011,220,5,2,3,2
gX-012,136,1,2,1,1
gX-013,1,1,1,1,1
gX-014,90,6,8,2,2
gX-015,576,9,6,3,2


## Siguiente peldaño

1. Cambia `ARQ` en el bloque 1: `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas`.
2. Ejecuta los bloques 3, 4 y 5.
3. La comparativa crece una fila. Lo que empeore, también se cuenta.

Y al congelar el baseline, en la terminal:

```
git tag baseline-congelado
git add resultados && git commit -m "Baseline congelado: 3 repeticiones"
```